# 03 — Real-Time Simulator
This notebook generates a fresh batch of sensor readings, power events, and renewable
generation data with **current timestamps** and appends them to the KQL Database.

Run this notebook on a schedule (e.g. every 5 minutes via pipeline) to simulate
real-time streaming data into the Eventhouse.

In [ ]:
import random
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

# Config: how many rows to generate per batch
SENSOR_BATCH = 500
EVENT_BATCH = 25
RENEWABLE_BATCH = 100

now = datetime.utcnow()
substations = [f"SUB-{i:03d}" for i in range(1, 26)]
regions = ["North", "South", "East", "West", "Central"]
sub_region = {s: regions[i % 5] for i, s in enumerate(substations)}

print(f"Generating batch at {now.strftime('%Y-%m-%d %H:%M:%S')} UTC")

In [ ]:
# Generate sensor readings spread across the last 5 minutes
sensor_rows = []
for i in range(SENSOR_BATCH):
    sub = random.choice(substations)
    ts = now - timedelta(seconds=random.randint(0, 300))
    hour = ts.hour
    base_v = 230.0 if random.random() > 0.03 else random.choice([210, 245, 250])
    voltage = round(random.gauss(base_v - (2 if 17 <= hour <= 21 else 0), 3.5), 2)
    freq = round(random.gauss(50.0, 0.05), 3)
    base_load = 15 + 10 * (1 if 8 <= hour <= 20 else 0) + 5 * (1 if 17 <= hour <= 21 else 0)
    sensor_rows.append(Row(
        reading_id=f"GS-RT-{now.strftime('%H%M%S')}-{i:04d}",
        timestamp=ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
        substation_id=sub,
        region=sub_region[sub],
        voltage_v=voltage,
        frequency_hz=freq,
        power_factor=round(random.uniform(0.85, 0.99), 3),
        load_mw=round(random.gauss(base_load, 4), 2),
        temperature_c=round(random.gauss(35 + 10 * (1 if 10 <= hour <= 16 else 0), 5), 1),
    ))

df_sensors = spark.createDataFrame(sensor_rows)
print(f"Generated {SENSOR_BATCH} sensor readings")

In [ ]:
# Generate power events
event_types = ["outage", "surge", "voltage_sag", "restoration", "equipment_fault", "overload"]
severities = ["low", "medium", "high", "critical"]
event_rows = []
for i in range(EVENT_BATCH):
    sub = random.choice(substations)
    etype = random.choice(event_types)
    ts = now - timedelta(seconds=random.randint(0, 300))
    dur = random.randint(1, 3600) if etype in ("outage", "equipment_fault") else random.randint(1, 120)
    affected = random.randint(0, 3000) if etype == "outage" else random.randint(0, 200)
    event_rows.append(Row(
        event_id=f"EVT-RT-{now.strftime('%H%M%S')}-{i:04d}",
        timestamp=ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
        substation_id=sub,
        region=sub_region[sub],
        event_type=etype,
        severity=random.choice(severities),
        duration_sec=dur,
        affected_customers=affected,
        resolved="true" if etype == "restoration" else random.choice(["true", "false"]),
    ))

df_events = spark.createDataFrame(event_rows)
print(f"Generated {EVENT_BATCH} power events")

In [ ]:
# Generate renewable readings
plants = [
    {"plant_id": f"SOLAR-{j:02d}", "plant_type": "solar", "capacity_mw": round(random.uniform(20, 80), 1)} for j in range(1, 9)
] + [
    {"plant_id": f"WIND-{j:02d}", "plant_type": "wind", "capacity_mw": round(random.uniform(30, 120), 1)} for j in range(1, 7)
] + [
    {"plant_id": f"HYDRO-{j:02d}", "plant_type": "hydro", "capacity_mw": round(random.uniform(50, 200), 1)} for j in range(1, 4)
]
weather_conditions = ["clear", "cloudy", "overcast", "rainy", "windy", "stormy"]
renewable_rows = []
for i in range(RENEWABLE_BATCH):
    plant = random.choice(plants)
    ts = now - timedelta(seconds=random.randint(0, 300))
    weather = random.choice(weather_conditions)
    hour = ts.hour
    if plant["plant_type"] == "solar":
        factor = max(0, (1 - abs(hour - 12) / 8)) * (0.3 if weather in ("cloudy", "overcast", "rainy") else 0.9)
    elif plant["plant_type"] == "wind":
        factor = random.uniform(0.2, 0.8) * (1.3 if weather in ("windy", "stormy") else 0.7)
    else:
        factor = random.uniform(0.5, 0.95)
    gen = round(plant["capacity_mw"] * factor * random.uniform(0.8, 1.1), 2)
    renewable_rows.append(Row(
        reading_id=f"RG-RT-{now.strftime('%H%M%S')}-{i:04d}",
        timestamp=ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
        plant_id=plant["plant_id"],
        plant_type=plant["plant_type"],
        generation_mw=max(0, gen),
        capacity_mw=plant["capacity_mw"],
        capacity_factor=round(max(0, gen) / plant["capacity_mw"], 3),
        weather=weather,
    ))

df_renewable = spark.createDataFrame(renewable_rows)
print(f"Generated {RENEWABLE_BATCH} renewable readings")

In [ ]:
# Ingest into KQL Database
KUSTO_CLUSTER = "{{EVENTHOUSE_URI}}"
KUSTO_DB = "{{KQL_DATABASE_NAME}}"

if KUSTO_CLUSTER and not KUSTO_CLUSTER.startswith("{{"):
    try:
        access_token = mssparkutils.credentials.getToken("kusto")
        
        df_sensors.write.format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", KUSTO_CLUSTER) \
            .option("kustoDatabase", KUSTO_DB) \
            .option("kustoTable", "GridSensors") \
            .option("accessToken", access_token) \
            .option("tableCreateOptions", "CreateIfNotExist") \
            .mode("Append") \
            .save()
        
        df_events.write.format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", KUSTO_CLUSTER) \
            .option("kustoDatabase", KUSTO_DB) \
            .option("kustoTable", "PowerEvents") \
            .option("accessToken", access_token) \
            .option("tableCreateOptions", "CreateIfNotExist") \
            .mode("Append") \
            .save()
        
        df_renewable.write.format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", KUSTO_CLUSTER) \
            .option("kustoDatabase", KUSTO_DB) \
            .option("kustoTable", "RenewableGeneration") \
            .option("accessToken", access_token) \
            .option("tableCreateOptions", "CreateIfNotExist") \
            .mode("Append") \
            .save()
        
        print(f"Batch ingested into KQL: {SENSOR_BATCH} sensors, {EVENT_BATCH} events, {RENEWABLE_BATCH} renewable")
        print(f"Total new rows: {SENSOR_BATCH + EVENT_BATCH + RENEWABLE_BATCH}")
    except Exception as e:
        print(f"KQL ingestion failed: {e}")
else:
    print("No Eventhouse URI — skipping KQL ingestion")